In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = Path("../../rqs/random_order")
TARGET_K = 8
N_SHUFFLES = 30

REPO_NAME_MAP = {
    "facebook_react": "React",
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "OpenCV",
    "tensorflow_tensorflow": "Tensorflow",
    "microsoft_vscode": "VsCode",
}

# ============================================================
# HELPERS
# ============================================================

def fmt_f1(x):
    """Format F1 with two decimals."""
    return f"{x:.2f}"

def fmt_pct(x):
    """Format percentage."""
    if x == np.inf:
        return r"$\infty$"
    return f"{x:.2f}\\%"

def latex_escape(s: str):
    return str(s).replace("_", r"\_")

# ============================================================
# STEP 1: LOAD + COMPUTE METRICS
# ============================================================

summary_rows = []

for repo_dir in BASE_DIR.iterdir():
    if not repo_dir.is_dir():
        continue

    repo = repo_dir.name
    models_dir = repo_dir / "models"

    if not models_dir.exists():
        continue

    for provider_dir in models_dir.iterdir():
        if not provider_dir.is_dir():
            continue

        for model_dir in provider_dir.iterdir():

            csv_path = model_dir / "logs" / "order_permutation_results_macro_f1.csv"

            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)
            df.columns = [c.strip() for c in df.columns]

            required_cols = {
                "FewShot_Count",
                "Shuffled_Macro_F1",
                "Original_Macro_F1",
            }

            if not required_cols.issubset(df.columns):
                continue

            df["FewShot_Count"] = pd.to_numeric(df["FewShot_Count"], errors="coerce")
            df["Shuffled_Macro_F1"] = pd.to_numeric(df["Shuffled_Macro_F1"], errors="coerce")
            df["Original_Macro_F1"] = pd.to_numeric(df["Original_Macro_F1"], errors="coerce")

            df = df.dropna(subset=["FewShot_Count"])
            df = df[df["FewShot_Count"] == TARGET_K]

            if df.empty:
                continue

            # Combine original + shuffled F1 values
            shuffled = df["Shuffled_Macro_F1"].dropna()
            original = df["Original_Macro_F1"].dropna().unique()

            f1 = pd.concat([
                shuffled,
                pd.Series(original)
            ])

            # Ensure range stays 0–1
            f1 = f1.clip(0, 1)

            min_f1 = f1.min()
            max_f1 = f1.max()
            mean_f1 = f1.mean()
            median_f1 = f1.median()

            delta_f1 = max_f1 - min_f1
            rel_impr = np.inf if min_f1 == 0 else (delta_f1 / min_f1) * 100

            summary_rows.append({
                "Repo": repo,
                "Model": model_dir.name,
                "min_f1": min_f1,
                "max_f1": max_f1,
                "mean_f1": mean_f1,
                "median_f1": median_f1,
                "delta_f1": delta_f1,
                "rel_impr": rel_impr,
            })

summary = pd.DataFrame(summary_rows)
summary = summary.sort_values(["Repo", "Model"]).reset_index(drop=True)

# ============================================================
# STEP 2: BUILD LATEX TABLE
# ============================================================

rows = []

for repo, repo_df in summary.groupby("Repo", sort=False):

    repo_name = REPO_NAME_MAP.get(repo, repo)
    repo_df = repo_df.reset_index(drop=True)

    for i, row in repo_df.iterrows():

        repo_cell = (
            rf"\multirow{{{len(repo_df)}}}{{*}}{{{repo_name}}}"
            if i == 0 else ""
        )

        rows.append(
            f"        {repo_cell} & {latex_escape(row['Model'])} & "
            f"{fmt_f1(row['min_f1'])} & "
            f"{fmt_f1(row['max_f1'])} & "
            f"{fmt_f1(row['mean_f1'])} & "
            f"{fmt_f1(row['median_f1'])} & "
            f"{fmt_f1(row['delta_f1'])} & "
            f"{fmt_pct(row['rel_impr'])} \\\\"
        )

    rows.append("        \\midrule")

rows = rows[:-1]

latex = "\n".join([
    r"\begin{table*}[t]",
    r"    \centering",
    rf"    \caption{{Macro-F1 range across order permutations per repository and model "
    rf"for few-shot count $k = {TARGET_K}$. Entries are computed over the original "
    rf"ordering and {N_SHUFFLES} shuffled orders.}}",
    rf"    \label{{tab:f1-range-order-shuffles-k{TARGET_K}}}",
    r"    \resizebox{\textwidth}{!}{%",
    r"        \begin{tabular}{llrrrrrr}",
    r"            \toprule",
    r"            \textbf{Repository} & \textbf{Model} & "
    r"\textbf{Min F1} & \textbf{Max F1} & "
    r"\textbf{Mean F1} & \textbf{Median F1} & "
    r"\textbf{$\Delta$F1} & \textbf{Rel. Impr.} \\",
    r"            \midrule",
    *rows,
    r"            \bottomrule",
    r"        \end{tabular}",
    r"    }",
    r"\end{table*}",
    "",
])

print(latex)

\begin{table*}[t]
    \centering
    \caption{Macro-F1 range across order permutations per repository and model for few-shot count $k = 8$. Entries are computed over the original ordering and 30 shuffled orders.}
    \label{tab:f1-range-order-shuffles-k8}
    \resizebox{\textwidth}{!}{%
        \begin{tabular}{llrrrrrr}
            \toprule
            \textbf{Repository} & \textbf{Model} & \textbf{Min F1} & \textbf{Max F1} & \textbf{Mean F1} & \textbf{Median F1} & \textbf{$\Delta$F1} & \textbf{Rel. Impr.} \\
            \midrule
        \multirow{4}{*}{Bitcoin} & Llama-3.1-8B-Instruct & 0.74 & 0.81 & 0.78 & 0.78 & 0.08 & 10.40\% \\
         & Mistral-7B-Instruct-v0.3 & 0.56 & 0.69 & 0.63 & 0.63 & 0.13 & 23.55\% \\
         & Qwen2.5-7B-Instruct & 0.73 & 0.83 & 0.77 & 0.76 & 0.10 & 13.41\% \\
         & gemma-3-12b-it & 0.72 & 0.81 & 0.75 & 0.75 & 0.09 & 12.71\% \\
        \midrule
        \multirow{4}{*}{React} & Llama-3.1-8B-Instruct & 0.61 & 0.79 & 0.75 & 0.75 & 0.18 & 29.35\% \\
  

In [2]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


# ============================================================
# CONFIG
# ============================================================

# RQ3: shuffled-order experiments
ORDER_BASE_DIR = Path("../../rqs/random_order")

# RQ1: original random-selection experiments
RANDOM_SELECTION_BASE_DIR = Path("../../rqs/random_selection")

TARGET_K = 8
N_SHUFFLES = 30

BOOTSTRAP_RUNS = 2000
SEED = 42

LABELS = [
    "bug",
    "feature",
    "question",
]

REPO_NAME_MAP = {
    "facebook_react": "React",
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "OpenCV",
    "tensorflow_tensorflow": "Tensorflow",
    "microsoft_vscode": "VS Code",
}


# ============================================================
# FORMATTING
# ============================================================

def fmt_f1(x):
    return f"{x:.2f}"


def fmt_f1_ci(point, low, high):
    return (
        f"{point:.2f} "
        f"[{low:.2f}, {high:.2f}]"
    )


def fmt_pct(x):
    if np.isinf(x):
        return r"$\infty$"

    return f"{x:.2f}\\%"


def latex_escape(s):
    return str(s).replace("_", r"\_")


# ============================================================
# INDIVIDUAL PARSING
# ============================================================

def parse_individual(value):

    if isinstance(value, list):
        return value

    return ast.literal_eval(str(value))


# ============================================================
# FIND PREDICTION FILE
# ============================================================

def find_prediction_file(
    logs_folder,
    individual,
):

    target = parse_individual(
        individual
    )

    prediction_files = list(
        logs_folder.rglob(
            "predictions_*.csv"
        )
    )

    for pred_file in prediction_files:

        try:
            first_row = pd.read_csv(
                pred_file,
                nrows=1,
            )
        except Exception:
            continue

        if "Individual" not in first_row.columns:
            continue

        try:
            candidate = parse_individual(
                first_row[
                    "Individual"
                ].iloc[0]
            )
        except Exception:
            continue

        if candidate == target:
            return pred_file


    raise FileNotFoundError(
        f"No prediction file found for "
        f"Individual={target}\n"
        f"under:\n{logs_folder}"
    )


# ============================================================
# PAIRED STRATIFIED BOOTSTRAP
# ============================================================

def paired_bootstrap(
    min_prediction_file,
    max_prediction_file,
    n_boot=BOOTSTRAP_RUNS,
    seed=SEED,
):

    df_min = pd.read_csv(
        min_prediction_file
    )

    df_max = pd.read_csv(
        max_prediction_file
    )


    # --------------------------------------------------------
    # Normalize labels
    # --------------------------------------------------------

    gt_min = (
        df_min["Ground_Truth"]
        .astype(str)
        .str.strip()
        .str.lower()
        .to_numpy()
    )

    gt_max = (
        df_max["Ground_Truth"]
        .astype(str)
        .str.strip()
        .str.lower()
        .to_numpy()
    )

    pred_min = (
        df_min["Prediction"]
        .astype(str)
        .str.strip()
        .str.lower()
        .to_numpy()
    )

    pred_max = (
        df_max["Prediction"]
        .astype(str)
        .str.strip()
        .str.lower()
        .to_numpy()
    )


    # --------------------------------------------------------
    # Sanity checks
    # --------------------------------------------------------

    assert len(gt_min) == len(gt_max), (
        "Prediction files do not contain "
        "the same number of test issues."
    )

    assert np.array_equal(
        gt_min,
        gt_max,
    ), (
        "Ground-truth ordering differs "
        "between the two prediction files."
    )

    gt = gt_min


    # --------------------------------------------------------
    # Point estimates
    # --------------------------------------------------------

    min_f1 = f1_score(
        gt,
        pred_min,
        labels=LABELS,
        average="macro",
        zero_division=0,
    )

    max_f1 = f1_score(
        gt,
        pred_max,
        labels=LABELS,
        average="macro",
        zero_division=0,
    )


    # --------------------------------------------------------
    # Stratified sampling indices
    # --------------------------------------------------------

    class_indices = {
        label: np.where(
            gt == label
        )[0]
        for label in LABELS
    }


    for label in LABELS:

        if len(
            class_indices[label]
        ) == 0:

            raise ValueError(
                f"No test instances found "
                f"for label '{label}'."
            )


    # --------------------------------------------------------
    # Bootstrap distributions
    # --------------------------------------------------------

    rng = np.random.default_rng(
        seed
    )

    min_distribution = np.empty(
        n_boot
    )

    max_distribution = np.empty(
        n_boot
    )

    delta_distribution = np.empty(
        n_boot
    )


    for i in range(n_boot):

        sample_indices = []

        for label in LABELS:

            pool = (
                class_indices[label]
            )

            draw = rng.choice(
                pool,
                size=len(pool),
                replace=True,
            )

            sample_indices.append(
                draw
            )


        sample_indices = (
            np.concatenate(
                sample_indices
            )
        )


        gt_sample = gt[
            sample_indices
        ]

        min_sample = pred_min[
            sample_indices
        ]

        max_sample = pred_max[
            sample_indices
        ]


        sample_min_f1 = f1_score(
            gt_sample,
            min_sample,
            labels=LABELS,
            average="macro",
            zero_division=0,
        )

        sample_max_f1 = f1_score(
            gt_sample,
            max_sample,
            labels=LABELS,
            average="macro",
            zero_division=0,
        )


        min_distribution[i] = (
            sample_min_f1
        )

        max_distribution[i] = (
            sample_max_f1
        )

        delta_distribution[i] = (
            sample_max_f1
            - sample_min_f1
        )


    # --------------------------------------------------------
    # 95% percentile confidence intervals
    # --------------------------------------------------------

    min_ci_low, min_ci_high = (
        np.percentile(
            min_distribution,
            [2.5, 97.5],
        )
    )

    max_ci_low, max_ci_high = (
        np.percentile(
            max_distribution,
            [2.5, 97.5],
        )
    )

    delta_ci_low, delta_ci_high = (
        np.percentile(
            delta_distribution,
            [2.5, 97.5],
        )
    )


    return {
        "min_f1": min_f1,
        "min_ci_low": min_ci_low,
        "min_ci_high": min_ci_high,

        "max_f1": max_f1,
        "max_ci_low": max_ci_low,
        "max_ci_high": max_ci_high,

        "delta_f1":
            max_f1 - min_f1,

        "delta_ci_low":
            delta_ci_low,

        "delta_ci_high":
            delta_ci_high,

        "bootstrap_positive_pct":
            (
                delta_distribution
                > 0
            ).mean() * 100,
    }


# ============================================================
# PROCESS ALL REPOSITORIES / MODELS
# ============================================================

summary_rows = []


for repo_dir in ORDER_BASE_DIR.iterdir():

    if not repo_dir.is_dir():
        continue


    repo = repo_dir.name

    models_dir = (
        repo_dir / "models"
    )


    if not models_dir.exists():
        continue


    for provider_dir in models_dir.iterdir():

        if not provider_dir.is_dir():
            continue


        provider = (
            provider_dir.name
        )


        for model_dir in provider_dir.iterdir():

            if not model_dir.is_dir():
                continue


            model = (
                model_dir.name
            )


            # =================================================
            # RQ3 ordering results
            # =================================================

            order_logs_folder = (
                model_dir
                / "logs"
            )

            results_file = (
                order_logs_folder
                / "order_permutation_results_macro_f1.csv"
            )


            if not results_file.exists():
                continue


            print()
            print("=" * 80)
            print(
                f"Processing "
                f"{repo} / {model}"
            )
            print("=" * 80)


            df = pd.read_csv(
                results_file
            )

            df.columns = [
                c.strip()
                for c in df.columns
            ]


            required_cols = {
                "FewShot_Count",
                "Original_Individual",
                "Shuffled_Individual",
                "Original_Macro_F1",
                "Shuffled_Macro_F1",
            }


            if not required_cols.issubset(
                df.columns
            ):

                missing = (
                    required_cols
                    - set(df.columns)
                )

                print(
                    f"Skipping because columns "
                    f"are missing: {missing}"
                )

                continue


            # =================================================
            # KEEP k = 8
            # =================================================

            df["FewShot_Count"] = (
                pd.to_numeric(
                    df["FewShot_Count"],
                    errors="coerce",
                )
            )

            df["Original_Macro_F1"] = (
                pd.to_numeric(
                    df["Original_Macro_F1"],
                    errors="coerce",
                )
            )

            df["Shuffled_Macro_F1"] = (
                pd.to_numeric(
                    df["Shuffled_Macro_F1"],
                    errors="coerce",
                )
            )


            df = df[
                df["FewShot_Count"]
                == TARGET_K
            ].copy()


            if df.empty:
                continue


            # =================================================
            # VERIFY EXPERIMENT DESIGN
            #
            # There should be exactly one Original_Individual
            # for this repo x model.
            # =================================================

            n_originals = (
                df["Original_Individual"]
                .nunique()
            )


            if n_originals != 1:

                raise ValueError(
                    f"{repo}/{model}: expected exactly "
                    f"1 Original_Individual, found "
                    f"{n_originals}."
                )


            original_individual = (
                df["Original_Individual"]
                .iloc[0]
            )

            original_f1 = (
                df["Original_Macro_F1"]
                .iloc[0]
            )


            print(
                "Original individual:",
                original_individual,
            )

            print(
                "Original F1:",
                original_f1,
            )

            print(
                "Number of shuffles:",
                len(df),
            )


            # =================================================
            # BUILD THE 31 ORDERINGS
            #
            # 1 original + 30 shuffled orders
            # =================================================

            configurations = []


            # -------------------------------------------------
            # Original ordering
            # -------------------------------------------------

            configurations.append({
                "source": "original",
                "individual":
                    original_individual,
                "f1":
                    original_f1,
            })


            # -------------------------------------------------
            # Shuffled orderings
            # -------------------------------------------------

            for _, row in df.iterrows():

                configurations.append({
                    "source": "shuffle",
                    "individual":
                        row[
                            "Shuffled_Individual"
                        ],
                    "f1":
                        row[
                            "Shuffled_Macro_F1"
                        ],
                })


            config_df = pd.DataFrame(
                configurations
            )


            config_df["f1"] = (
                config_df["f1"]
                .clip(0, 1)
            )


            if len(config_df) != (
                N_SHUFFLES + 1
            ):

                print(
                    f"WARNING: expected "
                    f"{N_SHUFFLES + 1} orderings "
                    f"but found "
                    f"{len(config_df)}."
                )


            # =================================================
            # MIN / MAX / MEAN / MEDIAN OVER ALL 31
            # =================================================

            min_idx = (
                config_df["f1"]
                .idxmin()
            )

            max_idx = (
                config_df["f1"]
                .idxmax()
            )


            min_row = (
                config_df.loc[
                    min_idx
                ]
            )

            max_row = (
                config_df.loc[
                    max_idx
                ]
            )


            min_f1 = (
                min_row["f1"]
            )

            max_f1 = (
                max_row["f1"]
            )

            mean_f1 = (
                config_df["f1"]
                .mean()
            )

            median_f1 = (
                config_df["f1"]
                .median()
            )

            delta_f1 = (
                max_f1 - min_f1
            )


            if min_f1 > 0:

                relative_difference = (
                    delta_f1
                    / min_f1
                    * 100
                )

            else:

                relative_difference = (
                    np.inf
                )


            print()
            print(
                f"Minimum: {min_f1:.4f} "
                f"({min_row['source']})"
            )

            print(
                f"Maximum: {max_f1:.4f} "
                f"({max_row['source']})"
            )


            # =================================================
            # DETERMINE WHERE TO SEARCH FOR PREDICTIONS
            # =================================================

            # RQ1 random-selection logs
            random_selection_logs = (
                RANDOM_SELECTION_BASE_DIR
                / repo
                / "models"
                / provider
                / model
                / "logs"
            )


            # -------------------------------------------------
            # MIN prediction file
            # -------------------------------------------------

            if min_row["source"] == "original":

                print(
                    "Minimum is ORIGINAL -> "
                    "searching random_selection"
                )

                min_prediction_file = (
                    find_prediction_file(
                        random_selection_logs,
                        min_row[
                            "individual"
                        ],
                    )
                )

            else:

                print(
                    "Minimum is SHUFFLE -> "
                    "searching random_order"
                )

                min_prediction_file = (
                    find_prediction_file(
                        order_logs_folder,
                        min_row[
                            "individual"
                        ],
                    )
                )


            # -------------------------------------------------
            # MAX prediction file
            # -------------------------------------------------

            if max_row["source"] == "original":

                print(
                    "Maximum is ORIGINAL -> "
                    "searching random_selection"
                )

                max_prediction_file = (
                    find_prediction_file(
                        random_selection_logs,
                        max_row[
                            "individual"
                        ],
                    )
                )

            else:

                print(
                    "Maximum is SHUFFLE -> "
                    "searching random_order"
                )

                max_prediction_file = (
                    find_prediction_file(
                        order_logs_folder,
                        max_row[
                            "individual"
                        ],
                    )
                )


            print(
                "Min predictions:",
                min_prediction_file,
            )

            print(
                "Max predictions:",
                max_prediction_file,
            )


            # =================================================
            # PAIRED BOOTSTRAP
            # =================================================

            bootstrap = paired_bootstrap(
                min_prediction_file,
                max_prediction_file,
            )


            # =================================================
            # SANITY CHECK
            #
            # F1 recomputed from predictions should match
            # the values in the ordering results CSV.
            # =================================================

            if not np.isclose(
                bootstrap["min_f1"],
                min_f1,
                atol=1e-6,
            ):

                print(
                    "WARNING: recomputed minimum "
                    f"F1 {bootstrap['min_f1']:.6f} "
                    f"does not match stored "
                    f"{min_f1:.6f}"
                )


            if not np.isclose(
                bootstrap["max_f1"],
                max_f1,
                atol=1e-6,
            ):

                print(
                    "WARNING: recomputed maximum "
                    f"F1 {bootstrap['max_f1']:.6f} "
                    f"does not match stored "
                    f"{max_f1:.6f}"
                )


            # =================================================
            # SAVE RESULT
            # =================================================

            summary_rows.append({

                "Repo":
                    repo,

                "Model":
                    model,

                "min_source":
                    min_row[
                        "source"
                    ],

                "min_f1":
                    bootstrap[
                        "min_f1"
                    ],

                "min_ci_low":
                    bootstrap[
                        "min_ci_low"
                    ],

                "min_ci_high":
                    bootstrap[
                        "min_ci_high"
                    ],

                "max_source":
                    max_row[
                        "source"
                    ],

                "max_f1":
                    bootstrap[
                        "max_f1"
                    ],

                "max_ci_low":
                    bootstrap[
                        "max_ci_low"
                    ],

                "max_ci_high":
                    bootstrap[
                        "max_ci_high"
                    ],

                "mean_f1":
                    mean_f1,

                "median_f1":
                    median_f1,

                "delta_f1":
                    bootstrap[
                        "delta_f1"
                    ],

                "delta_ci_low":
                    bootstrap[
                        "delta_ci_low"
                    ],

                "delta_ci_high":
                    bootstrap[
                        "delta_ci_high"
                    ],

                "rel_diff":
                    relative_difference,

                "bootstrap_positive_pct":
                    bootstrap[
                        "bootstrap_positive_pct"
                    ],
            })


# ============================================================
# CREATE SUMMARY DATAFRAME
# ============================================================

summary = pd.DataFrame(
    summary_rows
)


summary = summary.sort_values(
    [
        "Repo",
        "Model",
    ]
).reset_index(
    drop=True
)


Processing bitcoin_bitcoin / gemma-3-12b-it
Original individual: [118, 209, 230, 273, 338, 439, 603, 618]
Original F1: 0.7278406598227604
Number of shuffles: 30

Minimum: 0.7219 (shuffle)
Maximum: 0.8137 (shuffle)
Minimum is SHUFFLE -> searching random_order
Maximum is SHUFFLE -> searching random_order
Min predictions: ..\..\rqs\random_order\bitcoin_bitcoin\models\google\gemma-3-12b-it\logs\generations\fs_8_shuffle_3\predictions_db7781e9-6e51-4685-8fdc-13a835b25a54.csv
Max predictions: ..\..\rqs\random_order\bitcoin_bitcoin\models\google\gemma-3-12b-it\logs\generations\fs_8_shuffle_25\predictions_641867fb-3e14-4397-8e24-ad58715d124c.csv

Processing bitcoin_bitcoin / Llama-3.1-8B-Instruct
Original individual: [118, 209, 230, 273, 338, 439, 603, 618]
Original F1: 0.7523384791309319
Number of shuffles: 30

Minimum: 0.7364 (shuffle)
Maximum: 0.8131 (shuffle)
Minimum is SHUFFLE -> searching random_order
Maximum is SHUFFLE -> searching random_order
Min predictions: ..\..\rqs\random_order\bi

In [3]:
summary

,Repo,Model,min_source,min_f1,min_ci_low,min_ci_high,max_source,max_f1,max_ci_low,max_ci_high,mean_f1,median_f1,delta_f1,delta_ci_low,delta_ci_high,rel_diff,bootstrap_positive_pct
0,bitcoin_bitcoin,Llama-3.1-8B-Instruct,shuffle,0.736444,0.627906,0.832799,shuffle,0.813051,0.716794,0.892047,0.779602,0.775092,0.076607,-0.008853,0.169232,10.402238,96.00
1,bitcoin_bitcoin,Mistral-7B-Instruct-v0.3,shuffle,0.559914,0.456913,0.657693,shuffle,0.691753,0.588650,0.789352,0.626391,0.626713,0.131839,0.065159,0.215760,23.546315,100.00
2,bitcoin_bitcoin,Qwen2.5-7B-Instruct,shuffle,0.730159,0.636464,0.813635,shuffle,0.828083,0.745690,0.906548,0.766666,0.764364,0.097923,0.043007,0.163632,13.411247,100.00
3,bitcoin_bitcoin,gemma-3-12b-it,shuffle,0.721945,0.616888,0.821670,shuffle,0.813671,0.713680,0.901006,0.751866,0.750318,0.091726,0.028673,0.164588,12.705381,100.00
4,facebook_react,Llama-3.1-8B-Instruct,shuffle,0.608033,0.470849,0.742463,shuffle,0.786508,0.653955,0.892967,0.747190,0.752226,0.178475,0.069529,0.284859,29.352925,100.00
5,facebook_react,Mistral-7B-Instruct-v0.3,shuffle,0.528514,0.395460,0.650660,shuffle,0.646788,0.508326,0.777977,0.579448,0.579172,0.118274,0.017022,0.231629,22.378664,99.05
6,facebook_react,Qwen2.5-7B-Instruct,shuffle,0.593182,0.451884,0.727969,shuffle,0.723551,0.585016,0.854086,0.659021,0.659173,0.130369,0.051272,0.218936,21.977918,100.00
7,facebook_react,gemma-3-12b-it,shuffle,0.656396,0.511583,0.786599,shuffle,0.793588,0.650794,0.905853,0.719258,0.720284,0.137192,0.053333,0.232951,20.900783,100.00
8,microsoft_vscode,Llama-3.1-8B-Instruct,shuffle,0.365388,0.335132,0.396249,shuffle,0.524055,0.483633,0.565019,0.446727,0.449573,0.158667,0.122435,0.197931,43.424376,100.00
9,microsoft_vscode,Mistral-7B-Instruct-v0.3,original,0.277806,0.254677,0.302931,shuffle,0.399081,0.363723,0.435282,0.330987,0.336383,0.121274,0.090135,0.153563,43.654303,100.00


In [4]:
# ============================================================
# BUILD LATEX TABLE
# ============================================================

rows = []


for repo, repo_df in summary.groupby(
    "Repo",
    sort=False,
):

    repo_name = (
        REPO_NAME_MAP.get(
            repo,
            repo,
        )
    )


    repo_df = (
        repo_df
        .reset_index(drop=True)
    )


    for i, row in repo_df.iterrows():

        repo_cell = (
            rf"\multirow{{{len(repo_df)}}}"
            rf"{{*}}{{{repo_name}}}"
            if i == 0
            else ""
        )


        rows.append(
            f"        {repo_cell} & "
            f"{latex_escape(row['Model'])} & "
            f"{fmt_f1_ci(row['min_f1'], row['min_ci_low'], row['min_ci_high'])} & "
            f"{fmt_f1_ci(row['max_f1'], row['max_ci_low'], row['max_ci_high'])} & "
            f"{fmt_f1(row['mean_f1'])} & "
            f"{fmt_f1(row['median_f1'])} & "
            f"{fmt_f1_ci(row['delta_f1'], row['delta_ci_low'], row['delta_ci_high'])} & "
            f"{fmt_pct(row['rel_diff'])} \\\\"
        )


    rows.append(
        r"        \midrule"
    )


# Remove final midrule
rows = rows[:-1]


# ============================================================
# LATEX
# ============================================================

latex = "\n".join([

    r"\begin{table*}[t!]",

    r"    \centering",

    rf"    \caption{{Macro-F1 variation across order permutations "
    rf"for few-shot count $k={TARGET_K}$. For each repository--model "
    rf"configuration, values correspond to the fixed demonstration set "
    rf"with the largest observed ordering range over the original ordering "
    rf"and {N_SHUFFLES} shuffled orders. Brackets report 95\% stratified "
    rf"paired-bootstrap confidence intervals based on "
    rf"{BOOTSTRAP_RUNS:,} resamples.}}",

    rf"    \label{{tab:f1-range-order-shuffles-k{TARGET_K}}}",

    r"    \footnotesize",

    r"    \resizebox{\textwidth}{!}{%",

    r"        \begin{tabular}{llccrrcr}",

    r"            \toprule",

    r"            \textbf{Repository} & "
    r"\textbf{Model} & "
    r"\textbf{Min F1 [95\% CI]} & "
    r"\textbf{Max F1 [95\% CI]} & "
    r"\textbf{Mean F1} & "
    r"\textbf{Median F1} & "
    r"\textbf{$\Delta$F1 [95\% CI]} & "
    r"\textbf{Rel. Diff.} \\",

    r"            \midrule",

    *rows,

    r"            \bottomrule",

    r"        \end{tabular}",

    r"    }",

    r"\end{table*}",

    "",
])

print("\n")
print(latex)



\begin{table*}[t!]
    \centering
    \caption{Macro-F1 variation across order permutations for few-shot count $k=8$. For each repository--model configuration, values correspond to the fixed demonstration set with the largest observed ordering range over the original ordering and 30 shuffled orders. Brackets report 95\% stratified paired-bootstrap confidence intervals based on 2,000 resamples.}
    \label{tab:f1-range-order-shuffles-k8}
    \footnotesize
    \resizebox{\textwidth}{!}{%
        \begin{tabular}{llccrrcr}
            \toprule
            \textbf{Repository} & \textbf{Model} & \textbf{Min F1 [95\% CI]} & \textbf{Max F1 [95\% CI]} & \textbf{Mean F1} & \textbf{Median F1} & \textbf{$\Delta$F1 [95\% CI]} & \textbf{Rel. Diff.} \\
            \midrule
        \multirow{4}{*}{Bitcoin} & Llama-3.1-8B-Instruct & 0.74 [0.63, 0.83] & 0.81 [0.72, 0.89] & 0.78 & 0.78 & 0.08 [-0.01, 0.17] & 10.40\% \\
         & Mistral-7B-Instruct-v0.3 & 0.56 [0.46, 0.66] & 0.69 [0.59, 0.79] & 0.63 &